# ByteTrack

In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 78.7 MB/s eta 0:00:00


In [2]:
import torch
print("Pytorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

Pytorch version: 2.11.0+cu128
GPU available: True


In [3]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.track(
    source= "/content/Cars.mp4",
    tracker="bytetrack.yaml",
    conf=0.4,
    iou=0.5,
    persist=True,
    save=True,
    device=0
)

print("Tracking completed.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 204ms
Prepared 1 package in 32ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r 

# DeepSORT

In [10]:
!pip install -q ultralytics deep-sort-realtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 108.3 MB/s eta 0:00:00


In [11]:
import torch

device = 0 if torch.cuda.is_available() else "cpu"

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [12]:
import cv2
import torch
from pathlib import Path
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort


# --------------------------------------------------
# Configuration
# --------------------------------------------------

MODEL_PATH = "yolo11n.pt"
OUTPUT_PATH = "/content/deepsort_output.mp4"

CONFIDENCE_THRESHOLD = 0.4

# None means detect and track all supported classes.
# Use [0] for people only.
# Use [2, 3, 5, 7] for vehicles.
CLASSES_TO_TRACK = None


# --------------------------------------------------
# Load YOLO detector
# --------------------------------------------------

model = YOLO(MODEL_PATH)

device = 0 if torch.cuda.is_available() else "cpu"


# --------------------------------------------------
# Initialize DeepSORT
# --------------------------------------------------

tracker = DeepSort(
    max_age=30,
    n_init=3,
    max_iou_distance=0.7,
    max_cosine_distance=0.2,
    nn_budget=100,
    embedder="mobilenet",
    half=torch.cuda.is_available(),
    bgr=True,
    embedder_gpu=torch.cuda.is_available(),
)


# --------------------------------------------------
# Open input video
# --------------------------------------------------
video_path = "/content/Cars.mp4"
capture = cv2.VideoCapture(video_path)

if not capture.isOpened():
    raise RuntimeError(f"Could not open video: {video_path}")

fps = capture.get(cv2.CAP_PROP_FPS)
width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))

if fps <= 0:
    fps = 30.0


# --------------------------------------------------
# Create output video
# --------------------------------------------------

writer = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height),
)

if not writer.isOpened():
    capture.release()
    raise RuntimeError("Could not create the output video.")


# Store previous center points for trajectories
track_history = {}

frame_number = 0


# --------------------------------------------------
# Process the video
# --------------------------------------------------

while True:
    success, frame = capture.read()

    if not success:
        break

    # Run YOLO detection
    results = model.predict(
        source=frame,
        conf=CONFIDENCE_THRESHOLD,
        classes=CLASSES_TO_TRACK,
        device=device,
        verbose=False,
    )

    result = results[0]

    # DeepSORT detections:
    # ([left, top, width, height], confidence, class_name)
    detections = []

    if result.boxes is not None:
        boxes = result.boxes.xyxy.cpu().numpy()
        confidences = result.boxes.conf.cpu().numpy()
        class_ids = result.boxes.cls.int().cpu().numpy()

        for box, confidence, class_id in zip(
            boxes,
            confidences,
            class_ids,
        ):
            x1, y1, x2, y2 = box

            width_box = x2 - x1
            height_box = y2 - y1

            class_name = model.names[int(class_id)]

            detections.append(
                (
                    [
                        float(x1),
                        float(y1),
                        float(width_box),
                        float(height_box),
                    ],
                    float(confidence),
                    class_name,
                )
            )

    # Update DeepSORT
    tracks = tracker.update_tracks(
        detections,
        frame=frame,
    )

    # Draw confirmed tracks
    for track in tracks:
        if not track.is_confirmed():
            continue

        track_id = track.track_id

        # Bounding box: left, top, right, bottom
        left, top, right, bottom = track.to_ltrb()

        left = int(left)
        top = int(top)
        right = int(right)
        bottom = int(bottom)

        class_name = track.get_det_class()

        if class_name is None:
            class_name = "object"

        center_x = int((left + right) / 2)
        center_y = int((top + bottom) / 2)

        # Store trajectory
        if track_id not in track_history:
            track_history[track_id] = []

        track_history[track_id].append((center_x, center_y))

        # Limit trajectory length
        if len(track_history[track_id]) > 50:
            track_history[track_id].pop(0)

        # Draw bounding box
        cv2.rectangle(
            frame,
            (left, top),
            (right, bottom),
            (0, 255, 0),
            2,
        )

        # Draw class name and tracking ID
        label = f"{class_name} ID: {track_id}"

        cv2.putText(
            frame,
            label,
            (left, max(top - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            2,
        )

        # Draw center point
        cv2.circle(
            frame,
            (center_x, center_y),
            4,
            (0, 0, 255),
            -1,
        )

        # Draw trajectory
        points = track_history[track_id]

        for index in range(1, len(points)):
            cv2.line(
                frame,
                points[index - 1],
                points[index],
                (255, 255, 255),
                2,
            )

    writer.write(frame)

    frame_number += 1

    if frame_number % 50 == 0:
        print(
            f"Processed {frame_number}/{total_frames} frames"
        )


capture.release()
writer.release()

print("DeepSORT tracking completed.")
print("Output saved at:", Path(OUTPUT_PATH))

Processed 50/718 frames
Processed 100/718 frames
Processed 150/718 frames
Processed 200/718 frames
Processed 250/718 frames
Processed 300/718 frames
Processed 350/718 frames
Processed 400/718 frames
Processed 450/718 frames
Processed 500/718 frames
Processed 550/718 frames
Processed 600/718 frames
Processed 650/718 frames
Processed 700/718 frames
DeepSORT tracking completed.
Output saved at: /content/deepsort_output.mp4
